In [1]:
from tqdm.auto import tqdm
import numpy as np

from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
import json

from openai import OpenAI
openai_client = OpenAI()

In [4]:
from minsearch import VectorSearch

texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)
vindex = VectorSearch(keyword_fields=["course"])

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
vindex.fit(X, documents)

  0%|          | 0/28 [00:00<?, ?it/s]

In [5]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [6]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while the program is still accepting submissions.'

In [7]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [8]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

## 07- SQLiteSearch

In [10]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

vs_index.fit(vectors, documents)

In [14]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [16]:
vs_index.close()